# 01: Download and preprocess data

Build analysis-ready **GPM IMERG** precipitation, **GLDAS** land-surface fields, and **GRACE/GRACE-FO mascons** for the arid GRACE study.

| Product | Role | Output |
|---------|------|--------|
| GPM IMERG Final daily (`GPM_3IMERGDF`) | Precipitation | `data/interim/gpm/*_resToM.zarr` (monthly sums, mm/month) |
| GLDAS CLSM / NOAH / VIC (1°, monthly) | SM, runoff, SWE | `data/interim/gldas/<model>/<model>_<TAG>.zarr` (one file per model, mm) |
| CSR / JPL / GSFC mascons | TWSA | `data/raw/grace/{csr,jpl,gsfc}/` |

**Study window (default):** 2002-01-01 to 2025-09-30 (covers notebook `03`).

**Time convention:** month-end timestamps (IMERG `resample(time="ME")`; GLDAS normalized to month-end on write).

**Still manual:** arid-boundary shapefiles under `data/processed/boundaries/` (see `data/README.md`).

Helpers: `src/download_data.py`.


In [1]:
# Project paths: run with cwd = repository root (or notebooks/ - auto-detected)
from pathlib import Path
import sys
import logging

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    candidate = PROJECT_ROOT.parent
    if (candidate / "src").is_dir():
        PROJECT_ROOT = candidate
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
for d in (RAW_DIR / "gpm", RAW_DIR / "gldas", INTERIM_DIR / "gpm", INTERIM_DIR / "gldas"):
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")
print(f"repo: {PROJECT_ROOT}")

%load_ext autoreload
%autoreload 2


repo: /mnt/d/codes/python/grace_ds/github


## Earthdata Login (required)

1. Create a free account at [URS Earthdata Login](https://urs.earthdata.nasa.gov).
2. Configure credentials for [`earthaccess`](https://earthaccess.readthedocs.io/), typically a `~/.netrc` entry (and `~/.dodsrc` if you use OPeNDAP). Guide: [Earthdata Login prerequisites](https://nasa-openscapes.github.io/earthdata-cloud-cookbook/get-started/earthdata-login.html).
3. The next cell calls `ensure_earthdata_login()` (interactive prompt is OK if `.netrc` is missing).

No tokens belong in this repository. Do not commit `.netrc` or passwords.


In [2]:
%%time
# Config - adjust window / FORCE only if needed
from download_data import (
    DEFAULT_START,
    DEFAULT_END,
    GLOBAL_BBOX,
    GLDAS_MODELS,
    GLDAS_PRODUCTS,
    download_grace_mascons,
    ensure_earthdata_login,
    get_resource_config,
    period_tag,
    resolve_grace_paths,
    run_imerg_pipeline,
    run_gldas_all,
    set_project_root,
    summarize_zarr,
)

set_project_root(PROJECT_ROOT)

START = DEFAULT_START          # "2002-01-01"
END = DEFAULT_END              # "2025-09-30"
BBOX = GLOBAL_BBOX
# FORCE=True rebuilds Zarrs and re-queries catalogs. Existing raw granules are still
# reused by earthaccess. Delete data/raw/gpm/... or data/raw/gldas/<model>/ for a full re-fetch.
# Re-run once with FORCE=True if you need GLDAS Zarrs rewritten with month-end time labels.
FORCE = False
# threads=None uses get_resource_config() (CPU-based). Override with ints or env:
# DOWNLOAD_THREADS, DASK_WORKERS
IMERG_THREADS = None
GLDAS_THREADS = None

TAG = period_tag(START, END)
res = get_resource_config()
print(f"period: {TAG}")
print(f"GLDAS: {len(GLDAS_MODELS)} models (vars {list(GLDAS_PRODUCTS)} per Zarr)")
print(f"FORCE: {FORCE}")
print(
    f"resources: download_threads={res['download_threads']}  "
    f"dask_workers={res['dask_workers']}  gpu={res['gpu']}"
)
print(f"  ({res['gpu_note']})")
auth = ensure_earthdata_login()


period: Jan2002_Sep2025
GLDAS: 3 models (vars ['SM', 'Q', 'SWE'] per Zarr)
FORCE: False
resources: download_threads=15  dask_workers=8  gpu=detected_unused
  (I/O-bound NetCDF->Zarr ETL; GPU not used)
Earthdata login: ok
CPU times: user 2.94 s, sys: 1.27 s, total: 4.21 s
Wall time: 6.36 s


## GPM IMERG Final daily to monthly Zarr

Downloads `GPM_3IMERGDF` granules and resamples them **directly** to month-end precipitation sums (mm/month) used by notebooks `02` and `03`. The daily product is not written to disk; any daily Zarr from earlier runs is removed automatically.

Expected output:
- `data/interim/gpm/GPM_3IMERGDF_<TAG>_resToM.zarr`


In [3]:
%%time
imerg = run_imerg_pipeline(
    raw_dir=RAW_DIR,
    interim_dir=INTERIM_DIR,
    start=START,
    end=END,
    bbox=BBOX,
    threads=IMERG_THREADS,
    force=FORCE,
)
print(summarize_zarr(imerg["monthly_zarr"]))


resources: download_threads=15  dask_workers=8  gpu=detected_unused (I/O-bound NetCDF->Zarr ETL; GPU not used)
IMERG monthly Zarr
  dir: data/interim/gpm/
  [ok] GPM_3IMERGDF_Jan2002_Sep2025_resToM.zarr
GPM_3IMERGDF_Jan2002_Sep2025_resToM.zarr: vars=['precipitation'] dims={'lat': 1800, 'lon': 3600, 'time': 285} time=2002-01-31 to 2025-09-30
CPU times: user 636 ms, sys: 69.3 ms, total: 706 ms
Wall time: 1.36 s


In [4]:
# OPTIONAL: reclaim disk space (kept commented on purpose).
# The monthly Zarr above is verified complete before anything is deleted.
# Raw daily IMERG granules are ~254 GB and are not needed again unless you
# rebuild with FORCE=True. Uncomment to free that space now.
# from download_data import remove_imerg_daily_granules
# removed = remove_imerg_daily_granules(RAW_DIR, INTERIM_DIR, start=START, end=END)
# print(f"Removed {removed['n_removed']} daily granules, freed {removed['freed_gb']:.1f} GB")


## GLDAS: one Zarr per model (SM + runoff + SWE)

One pass downloads each model's monthly NetCDFs, then writes a **single Zarr per model** holding all three analysis variables. Time is normalized to month-end on write.

Variables (matches notebook `03`):
- **Soil moisture** -> `SoilMoist_P_inst` (CLSM) or `sm_total` = sum of layers (NOAH / VIC)
- **Total runoff** -> `total_runoff` (`Qs_acc` + `Qsb_acc`)
- **SWE** -> `SWE_inst`

Units in the Zarr are mm (notebook 03 converts to cm with `/10`).


In [5]:
%%time
gldas = run_gldas_all(
    raw_dir=RAW_DIR,
    interim_dir=INTERIM_DIR,
    models=GLDAS_MODELS,
    start=START,
    end=END,
    bbox=BBOX,
    threads=GLDAS_THREADS,
    force=FORCE,
)


GLDAS (3 models, SM+Q+SWE)
  dir: data/interim/gldas/
  [ok] GLDAS_CLSM10_M/GLDAS_CLSM10_M_Jan2002_Sep2025.zarr
  [ok] GLDAS_NOAH10_M/GLDAS_NOAH10_M_Jan2002_Sep2025.zarr
  [ok] GLDAS_VIC10_M/GLDAS_VIC10_M_Jan2002_Sep2025.zarr
CPU times: user 45.8 ms, sys: 15.6 ms, total: 61.4 ms
Wall time: 532 ms


In [6]:
# OPTIONAL: reclaim disk space (kept commented on purpose).
# Each model Zarr above is verified complete before its raw granules are deleted.
# Raw GLDAS NetCDFs are ~1.7 GB total across CLSM/NOAH/VIC and are not needed
# again unless you rebuild with FORCE=True. Uncomment to free that space now.
# from download_data import remove_gldas_raw_granules, GLDAS_MODELS
# for model in GLDAS_MODELS:
#     removed = remove_gldas_raw_granules(model, RAW_DIR, INTERIM_DIR, start=START, end=END)
#     print(f"{model}: removed {removed['n_removed']} granules, freed {removed['freed_gb']:.1f} GB")


## GRACE / GRACE-FO mascons (CSR, JPL, GSFC)

Downloads the three mascon solutions used by notebooks `02` and `03`, plus the CSR land mask.

| Center | How | Stable filename tokens |
|--------|-----|------------------------|
| CSR | HTTPS from UTCSR page | `all-corrections`, `LandMask` |
| JPL CRI | Earthdata / `earthaccess` | `MSCNv04CRI` |
| GSFC half-degree OBP | HTTPS from GSFC page | `halfdegree` + `obp` |

Date spans in filenames change when centers update products; notebooks resolve files by the tokens above (newest match wins).


In [7]:
%%time
grace_paths = download_grace_mascons(raw_dir=RAW_DIR, force=FORCE)
resolved = resolve_grace_paths(RAW_DIR)
assert set(resolved) == {"csr", "csr_mask", "jpl", "gsfc"}


GRACE mascons
  dir: data/raw/grace/
  [ok] csr/CSR_GRACE_GRACE-FO_RL0603_Mascons_all-corrections.nc
  [ok] csr/CSR_GRACE_GRACE-FO_RL06_Mascons_v02_LandMask.nc
  [ok] jpl/GRCTellus.JPL.200204_202605.GLO.RL06.3M.MSCNv04CRI.nc
  [ok] gsfc/gsfc.glb_.200204_202603_rl06v2.0_obp-ice6gd_halfdegree.nc
CPU times: user 6.41 ms, sys: 18.7 ms, total: 25.1 ms
Wall time: 212 ms


## Sanity check

Confirm time coverage for precipitation / GLDAS and that GRACE paths resolve.


In [8]:
from download_data import gldas_zarr_path, resolve_grace_paths

def _nb_rel(path):
    try:
        return str(Path(path).resolve().relative_to(PROJECT_ROOT)).replace("\\", "/")
    except ValueError:
        return Path(path).name

monthly_precip = INTERIM_DIR / "gpm" / f"GPM_3IMERGDF_{TAG}_resToM.zarr"
example_gldas = gldas_zarr_path(INTERIM_DIR, "GLDAS_NOAH10_M", START, END)

print(summarize_zarr(monthly_precip))
print(summarize_zarr(example_gldas))
print("GRACE files:")
for key, path in resolve_grace_paths(RAW_DIR).items():
    print(f"  [{key}] {_nb_rel(path)}  ({path.stat().st_size/1e6:.0f} MB)")
print("Done. Next: run notebooks 02 / 03.")


GPM_3IMERGDF_Jan2002_Sep2025_resToM.zarr: vars=['precipitation'] dims={'lat': 1800, 'lon': 3600, 'time': 285} time=2002-01-31 to 2025-09-30
GLDAS_NOAH10_M_Jan2002_Sep2025.zarr: vars=['SWE_inst', 'sm_total', 'total_runoff'] dims={'time': 285, 'lat': 150, 'lon': 360} time=2002-01-01 to 2025-09-01
GRACE files:
  [csr] data/raw/grace/csr/CSR_GRACE_GRACE-FO_RL0603_Mascons_all-corrections.nc  (112 MB)
  [csr_mask] data/raw/grace/csr/CSR_GRACE_GRACE-FO_RL06_Mascons_v02_LandMask.nc  (4 MB)
  [jpl] data/raw/grace/jpl/GRCTellus.JPL.200204_202605.GLO.RL06.3M.MSCNv04CRI.nc  (46 MB)
  [gsfc] data/raw/grace/gsfc/gsfc.glb_.200204_202603_rl06v2.0_obp-ice6gd_halfdegree.nc  (531 MB)
Done. Next: run notebooks 02 / 03.
